# MidMamba Full Training Run (Colab)

End-to-end Mamba-2 PPO training on March 2025 RTH Databento MBP-10 data.

**Hardware target**: RTX PRO 6000 (96 GB VRAM, 48 Workers).  
**Optimizations**: 
- **Vectorized Rollouts**: 48 parallel environments using `AsyncVectorEnv`.
- **AMP (bfloat16)**: High-speed mixed precision optimized for Blackwell/Ada architectures.
- **Zero-Copy Resets**: Pre-extracted MBP-10 arrays to minimize environment overhead.

**Data source**: `.dbn.zst` files already in Google Drive at `/content/drive/MyDrive/midmamba/data/march2025/`.  
**Outputs**: Checkpoints and evaluation results written to `/content/drive/MyDrive/midmamba/`.

**Workflow**:  
1. Mount Drive and copy repo to local SSD for fast I/O  
2. Install dependencies (including `mamba-ssm` for GPU)  
3. Load March RTH data via chunked DBN decoder  
4. Train Mamba-2 PPO agent  
5. Evaluate against Immediate / TWAP / Almgren-Chriss baselines  
6. Save everything to Drive

In [ ]:
!pip install -U jupyter_client

In [ ]:
import os
import multiprocessing
os.environ["PYTHONWARNINGS"] = "ignore::DeprecationWarning,ignore::UserWarning"
multiprocessing.set_start_method("forkserver", force=True)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import shutil
import os
import sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/midmamba")
REPO_ROOT = Path("/content/midmamba")

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
shutil.copytree(DRIVE_ROOT, REPO_ROOT, symlinks=True)

src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.chdir(REPO_ROOT)

print(f"repo:  {REPO_ROOT}")
print(f"drive: {DRIVE_ROOT}")

In [ ]:
!pip install -q -e "/content/midmamba[dev]"

# Build mamba-ssm + causal-conv1d for Blackwell (sm_120)
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "12.0"
!pip install -U -q pip setuptools wheel ninja packaging
!pip install -q causal-conv1d --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation

import torch
print(f"torch {torch.__version__}  CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)
    print(f"VRAM: {vram / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected -- training will be slow")

In [ ]:
# ── All tunables in one place ──────────────────────────────────────────

BACKEND        = "mamba"       # "mamba" for GPU, "gru" for CPU fallback
D_MODEL        = 192
N_LAYERS       = 3
SEQ_LEN        = 128
SPATIAL_STEM   = True          # bid/ask-aware LOBSpatialStem vs flat linear
DROPOUT        = 0.1
NUM_ENVS       = 48            # match CPU core count for async vectorized rollout
USE_AMP        = True          # bfloat16 mixed precision
ROLLOUT_STEPS  = 512           # 512 × 48 envs = 24K steps per update
UPDATES        = 2000
PPO_EPOCHS     = 4             # CleanRL/SB3 standard; 4 × 6 = 24 grad steps/update
MINIBATCH_SIZE = 4096           # 24K / 4096 = 6 grad steps per epoch
CLIP_COEF      = 0.1           # tighter clip — 138-dim obs needs less freedom than MuJoCo
MAX_GRAD_NORM  = 0.5
ENTROPY_COEF   = 0.001          # pre-tanh entropy; log_std clamp does heavy lifting
TARGET_KL      = 0.01          # KL early-stop threshold (half of PPO default)
KL_GRACE_EPOCHS = 0            # no grace: stop immediately if KL exceeds 1.5×target
EXECUTION_STEPS   = 360          # 360 × 5s = 30-min horizon
PARENT_QUANTITY   = 100_000.0
LR             = 3e-5          # lower LR for high-dim Mamba (1e-4 diverges)
LR_SCHEDULE    = "linear"     # linear decay to 0 (MuJoCo standard); also: "constant", "cosine"
LR_WARMUP      = 10           # short warmup — reach peak LR quickly
GAMMA          = 0.995        # longer horizon for execution (matches RL-Exec)
GAE_LAMBDA     = 0.95
TERMINAL_PENALTY_BPS = 100.0  # legacy fallback (unused with Path C reward)
FILL_MODEL     = "random"     # domain randomization across fill assumptions

# ── Multi-objective volatility-scaled reward (Path C) ─────────────────
BETA_IS         = 1.0         # weight on implementation shortfall per-step
BETA_SCHEDULE   = 0.1         # soft schedule guide (not dominant — IS is the objective)
BETA_COMPLETION = 1.0         # σ√T-scaled terminal penalty (needs to be felt)
REWARD_CLIP     = 5.0         # hard clip on per-step reward (0 = no clip)
NORM_REWARD     = True        # normalize discounted returns — essential for long episodes

# ── Transaction costs ─────────────────────────────────────────────────
TAKER_FEE_BPS   = 2.0         # taker (market order) fee — CME NQ ~1-2 bps
MAKER_REBATE_BPS = 0.5        # maker (limit order) rebate
SIDE           = "buy"
SEED           = 1
CHECKPOINT_EVERY = 100
EVAL_EPISODES    = 200
TWAP_SLICES      = 100
AC_RISK_AVERSION     = 1e-6
AC_VOLATILITY        = 0.02
AC_TEMPORARY_IMPACT  = 1.0

# Data loading — training
DBN_GLOB       = "data/march2025/*.dbn.zst"
CHUNK_ROWS     = 500_000
MIN_LOADER_ROWS = 1_000       # validation floor (all files are always loaded)
MAX_CHUNKS     = None         # None = read all chunks from all files
RTH_ONLY       = True
RTH_START      = "09:30:00"
RTH_END        = "16:00:00"
RESAMPLE_FREQ  = "5s"         # 5-second fixed cadence (360 steps × 5s = 30 min)

# Data loading — evaluation (out-of-sample)
EVAL_DBN_GLOB  = "data/october2025/*.dbn.zst"  # separate test set

# Output paths (Drive-backed)
DRIVE_OUT       = Path("/content/drive/MyDrive/midmamba")
CHECKPOINT_PATH = DRIVE_OUT / "checkpoints" / "mamba_ppo.pt"
RESULTS_DIR     = DRIVE_OUT / "results"

CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={DEVICE}  backend={BACKEND}  updates={UPDATES}")

In [ ]:
dbn_files = sorted(REPO_ROOT.glob(DBN_GLOB))
print(f"found {len(dbn_files)} DBN files")
if dbn_files:
    print(f"  first: {dbn_files[0].name}")
    print(f"  last:  {dbn_files[-1].name}")
else:
    raise FileNotFoundError(
        f"no .dbn.zst files matched {DBN_GLOB} under {REPO_ROOT}.\n"
        "Make sure your Drive contains the data at /MyDrive/midmamba/data/march2025/"
    )

In [ ]:
!cd /content/midmamba && python -m pytest tests -q

## Training

In [ ]:
import gymnasium as gym
from midmamba.data import MBP10WindowLoader
from midmamba.env import MidMambaExecutionEnv
from midmamba.rl import VecNormalize

def _progress(info):
    print(
        f"  file={info.get('file_index', 1)} "
        f"chunk={info['chunk_index']} "
        f"decoded={info['decoded_rows']:,} "
        f"kept={info['kept_rows']:,}",
        end="\r",
    )

print("loading DBN data (chunked)...")
loader = MBP10WindowLoader.from_dbn_files_chunks(
    dbn_files,
    chunk_rows=CHUNK_ROWS,
    min_rows=MIN_LOADER_ROWS,
    max_chunks=MAX_CHUNKS,
    resample_freq=RESAMPLE_FREQ,
    rth_start=RTH_START if RTH_ONLY else None,
    rth_end=RTH_END if RTH_ONLY else None,
    seed=SEED,
    progress_callback=_progress,
)
print(f"\nloader ready: {loader.n_rows:,} rows, {loader.n_features} features")
print(f"feature sample: {loader.feature_names[:5]} ...")

_reward_kwargs = dict(
    beta_is=BETA_IS,
    beta_schedule=BETA_SCHEDULE,
    beta_completion=BETA_COMPLETION,
    reward_clip=REWARD_CLIP,
    taker_fee_bps=TAKER_FEE_BPS,
    maker_rebate_bps=MAKER_REBATE_BPS,
)

def _make_env(seed_offset: int):
    def _init():
        env_loader = MBP10WindowLoader(
            loader.features, loader.raw_lob,
            feature_names=loader.feature_names,
            seed=42 + seed_offset,
        )
        return MidMambaExecutionEnv(
            env_loader,
            execution_steps=EXECUTION_STEPS,
            initial_inventory=PARENT_QUANTITY,
            side=SIDE,
            fill_model=FILL_MODEL,
            terminal_penalty_bps=TERMINAL_PENALTY_BPS,
            **_reward_kwargs,
        )
    return _init

if NUM_ENVS > 1:
    raw_vec_env = gym.vector.AsyncVectorEnv([_make_env(i) for i in range(NUM_ENVS)])
    vec_env = VecNormalize(raw_vec_env, norm_obs=True, norm_reward=NORM_REWARD, gamma=GAMMA)
    print(f"vec_env: {NUM_ENVS} envs (async, {os.cpu_count()} CPU cores) + VecNormalize(obs{'+rew' if NORM_REWARD else ''})")
    print(f"  obs_space={raw_vec_env.single_observation_space.shape}  action_space={raw_vec_env.single_action_space.shape}")
    print(f"  reward: β_is={BETA_IS} β_sched={BETA_SCHEDULE} β_comp={BETA_COMPLETION} clip={REWARD_CLIP}")
else:
    raw_vec_env = MidMambaExecutionEnv(
        loader,
        execution_steps=EXECUTION_STEPS,
        initial_inventory=PARENT_QUANTITY,
        side=SIDE,
        fill_model=FILL_MODEL,
        terminal_penalty_bps=TERMINAL_PENALTY_BPS,
        **_reward_kwargs,
    )
    vec_env = VecNormalize(raw_vec_env, norm_obs=True, norm_reward=NORM_REWARD, gamma=GAMMA)
    print(f"env obs_space={raw_vec_env.observation_space.shape}  action_space={raw_vec_env.action_space.shape}  + VecNormalize(obs{'+rew' if NORM_REWARD else ''})")

single_env = MidMambaExecutionEnv(
    loader,
    execution_steps=EXECUTION_STEPS,
    initial_inventory=PARENT_QUANTITY,
    side=SIDE,
    fill_model=FILL_MODEL,
    terminal_penalty_bps=TERMINAL_PENALTY_BPS,
    **_reward_kwargs,
)
n_obs = single_env.observation_space.shape[0]

In [ ]:
import numpy as np
from midmamba.models import LOBMambaRLExecutionAgent
from midmamba.rl import make_lr_lambda

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

n_features = n_obs
feature_names = loader.feature_names + ["remaining_time", "remaining_inventory", "last_fill_frac", "twap_deviation"]
agent = LOBMambaRLExecutionAgent(
    n_features=n_features, d_model=D_MODEL, action_dim=2, action_mode="continuous",
    n_layers=N_LAYERS, backend=BACKEND, spatial_stem=SPATIAL_STEM,
    feature_names=feature_names if SPATIAL_STEM else None, dropout=DROPOUT,
).to(DEVICE)
print(f"agent parameters: {sum(p.numel() for p in agent.parameters()):,}")

try:
    agent = torch.compile(agent, mode="default")
    print("torch.compile enabled (default mode, no CUDA graphs)")
except Exception as e:
    print(f"torch.compile skipped: {e}")

optimizer = torch.optim.Adam(agent.parameters(), lr=LR, eps=1e-5)
_lr_lambda = make_lr_lambda(LR_SCHEDULE, UPDATES, warmup_updates=LR_WARMUP)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)
print(f"lr_schedule={LR_SCHEDULE}  warmup={LR_WARMUP}  num_envs={NUM_ENVS}  amp={USE_AMP}")

run_config = {
    "backend": BACKEND, "d_model": D_MODEL, "n_layers": N_LAYERS,
    "spatial_stem": SPATIAL_STEM, "dropout": DROPOUT,
    "n_features": n_features, "seq_len": SEQ_LEN,
    "num_envs": NUM_ENVS, "use_amp": USE_AMP,
    "rollout_steps": ROLLOUT_STEPS, "updates": UPDATES,
    "ppo_epochs": PPO_EPOCHS, "minibatch_size": MINIBATCH_SIZE,
    "max_grad_norm": MAX_GRAD_NORM,
    "execution_steps": EXECUTION_STEPS, "parent_quantity": PARENT_QUANTITY,
    "lr": LR, "lr_schedule": LR_SCHEDULE, "lr_warmup": LR_WARMUP,
    "clip_coef": CLIP_COEF, "target_kl": TARGET_KL, "kl_grace_epochs": KL_GRACE_EPOCHS,
    "gamma": GAMMA, "gae_lambda": GAE_LAMBDA,
    "fill_model": FILL_MODEL, "side": SIDE, "seed": SEED,
    "loader_rows": loader.n_rows, "device": str(DEVICE),
    "beta_is": BETA_IS, "beta_schedule": BETA_SCHEDULE,
    "beta_completion": BETA_COMPLETION, "reward_clip": REWARD_CLIP,
    "taker_fee_bps": TAKER_FEE_BPS, "maker_rebate_bps": MAKER_REBATE_BPS,
}

In [ ]:
import time
from midmamba.rl import collect_rollout, ppo_update

SMOKE_UPDATES = 50
SMOKE_WARMUP  = 5   # short warmup for smoke (full run uses LR_WARMUP=50)
smoke_history = []
t0 = time.time()

smoke_optimizer = torch.optim.Adam(agent.parameters(), lr=LR, eps=1e-5)
smoke_scheduler = torch.optim.lr_scheduler.LambdaLR(
    smoke_optimizer,
    lambda u: min(1.0, float(u + 1) / float(max(1, SMOKE_WARMUP))),
)

eff_batch = ROLLOUT_STEPS * NUM_ENVS
print(f"smoke run: {SMOKE_UPDATES} updates, rollout={ROLLOUT_STEPS}, num_envs={NUM_ENVS}, eff_batch={eff_batch}, amp={USE_AMP}")
for u in range(1, SMOKE_UPDATES + 1):
    batch, rm = collect_rollout(
        vec_env, agent, rollout_steps=ROLLOUT_STEPS, seq_len=SEQ_LEN,
        device=DEVICE, gamma=GAMMA, gae_lambda=GAE_LAMBDA, use_amp=USE_AMP,
    )
    tm = ppo_update(
        agent, smoke_optimizer, batch,
        epochs=PPO_EPOCHS, minibatch_size=MINIBATCH_SIZE, clip_coef=CLIP_COEF,
        max_grad_norm=MAX_GRAD_NORM, entropy_coef=ENTROPY_COEF,
        target_kl=TARGET_KL, kl_grace_epochs=KL_GRACE_EPOCHS, use_amp=USE_AMP,
    )
    smoke_scheduler.step()
    smoke_history.append({"update": u, **rm, **tm})
    if u % 10 == 0 or u == 1:
        print(
            f"  [{u:>3}/{SMOKE_UPDATES}] "
            f"rew={rm['rollout_reward_mean']:+.4f}  "
            f"ep={rm['completed_episodes']:.0f}  "
            f"loss={tm['loss']:.5f}  "
            f"vloss={tm['value_loss']:.4f}  "
            f"ent={tm['entropy']:.4f}  "
            f"kl={tm['approx_kl']:.4f}  "
            f"clip={tm['clip_fraction']:.2f}  "
            f"gnorm={tm['grad_norm']:.3f}  "
            f"ep_used={tm['epochs_used']}  "
            f"({time.time()-t0:.0f}s)"
        )

r0 = smoke_history[0]["rollout_reward_mean"]
r_last = np.mean([h["rollout_reward_mean"] for h in smoke_history[-10:]])
print(f"\nsmoke result: reward {r0:+.4f} -> {r_last:+.4f} ({'improving' if r_last > r0 else 'NOT improving'})")
print(f"elapsed: {time.time()-t0:.0f}s  ({(time.time()-t0)/SMOKE_UPDATES:.1f}s/update)")
print(f"est. full run: {(time.time()-t0)/SMOKE_UPDATES * UPDATES / 3600:.1f} hours for {UPDATES} updates")

# Keep smoke weights as warm start (avoids recompile penalty and wastes no training)
optimizer = torch.optim.Adam(agent.parameters(), lr=LR, eps=1e-5)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)
print("optimizer and scheduler reset -- smoke weights kept as warm start")

In [ ]:
import json
from midmamba.rl import train_loop

history = train_loop(
    vec_env, agent, optimizer, scheduler,
    updates=UPDATES, rollout_steps=ROLLOUT_STEPS, seq_len=SEQ_LEN,
    ppo_epochs=PPO_EPOCHS, minibatch_size=MINIBATCH_SIZE,
    clip_coef=CLIP_COEF, max_grad_norm=MAX_GRAD_NORM, entropy_coef=ENTROPY_COEF,
    target_kl=TARGET_KL, kl_grace_epochs=KL_GRACE_EPOCHS,
    gamma=GAMMA, gae_lambda=GAE_LAMBDA, use_amp=USE_AMP, device=DEVICE,
    checkpoint_every=CHECKPOINT_EVERY, checkpoint_path=str(CHECKPOINT_PATH),
    run_config=run_config, vec_env_state_fn=vec_env.get_state,
)

# Final checkpoint
torch.save(
    {"model": agent.state_dict(), "optimizer": optimizer.state_dict(),
     "scheduler": scheduler.state_dict(), "config": run_config, "update": UPDATES,
     "vec_normalize": vec_env.get_state()},
    CHECKPOINT_PATH,
)

metrics_path = RESULTS_DIR / "ppo_training_metrics.json"
metrics_path.write_text(json.dumps({"config": run_config, "history": history}, indent=2, default=str))
print(f"checkpoint: {CHECKPOINT_PATH}")
print(f"metrics:    {metrics_path}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
upd = [r["update"] for r in history]

axes[0, 0].plot(upd, [r["rollout_reward_mean"] for r in history], alpha=0.3)
window = min(50, len(history) // 5 or 1)
if len(history) > window:
    smooth = np.convolve([r["rollout_reward_mean"] for r in history], np.ones(window)/window, mode="valid")
    axes[0, 0].plot(upd[window-1:], smooth, linewidth=2)
axes[0, 0].set_title("Rollout Reward Mean")
axes[0, 0].set_xlabel("Update")
axes[0, 0].grid(True)

axes[0, 1].plot(upd, [r["lr"] for r in history])
axes[0, 1].set_title("Learning Rate")
axes[0, 1].set_xlabel("Update")
axes[0, 1].ticklabel_format(axis="y", style="sci", scilimits=(-3, -3))
axes[0, 1].grid(True)

axes[1, 0].plot(upd, [r["policy_loss"] for r in history])
axes[1, 0].set_title("Policy Loss")
axes[1, 0].set_xlabel("Update")
axes[1, 0].grid(True)

axes[1, 1].plot(upd, [r["value_loss"] for r in history])
axes[1, 1].set_title("Value Loss")
axes[1, 1].set_xlabel("Update")
axes[1, 1].grid(True)

axes[2, 0].plot(upd, [r["entropy"] for r in history])
axes[2, 0].set_title("Entropy")
axes[2, 0].set_xlabel("Update")
axes[2, 0].grid(True)

# Episode reward: filter out updates with no completed episodes
ep_upd = [r["update"] for r in history if r["completed_episodes"] > 0]
ep_rew = [r["episode_reward_mean"] for r in history if r["completed_episodes"] > 0]
axes[2, 1].plot(ep_upd, ep_rew, alpha=0.3)
if len(ep_rew) > window:
    smooth_ep = np.convolve(ep_rew, np.ones(window)/window, mode="valid")
    axes[2, 1].plot(ep_upd[window-1:], smooth_ep, linewidth=2)
axes[2, 1].set_title("Episode Reward Mean (non-zero only)")
axes[2, 1].set_xlabel("Update")
axes[2, 1].grid(True)

axes[3, 0].plot(upd, [r.get("approx_kl", 0) for r in history])
axes[3, 0].axhline(y=0.02, color="r", linestyle="--", alpha=0.5, label="target_kl")
axes[3, 0].set_title("Approx KL")
axes[3, 0].set_xlabel("Update")
axes[3, 0].legend()
axes[3, 0].grid(True)

axes[3, 1].plot(upd, [r.get("clip_fraction", 0) for r in history])
axes[3, 1].set_title("Clip Fraction")
axes[3, 1].set_xlabel("Update")
axes[3, 1].grid(True)

plt.tight_layout()
plot_path = RESULTS_DIR / "training_curves.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"saved: {plot_path}")

## Evaluation

In [ ]:
# Load out-of-sample evaluation data (separate from training)
eval_dbn_files = sorted(REPO_ROOT.glob(EVAL_DBN_GLOB))
if eval_dbn_files:
    print(f"loading eval data: {len(eval_dbn_files)} files from {EVAL_DBN_GLOB}")
    eval_loader = MBP10WindowLoader.from_dbn_files_chunks(
        eval_dbn_files,
        chunk_rows=CHUNK_ROWS,
        min_rows=max(EXECUTION_STEPS, TWAP_SLICES + 1),
        resample_freq=RESAMPLE_FREQ,
        rth_start=RTH_START if RTH_ONLY else None,
        rth_end=RTH_END if RTH_ONLY else None,
        seed=SEED + 1000,
    )
    print(f"eval loader: {eval_loader.n_rows:,} rows (out-of-sample)")
else:
    print(f"WARNING: no eval files found at {EVAL_DBN_GLOB} -- falling back to training data (in-sample)")
    eval_loader = loader

In [ ]:
from midmamba.eval import run_immediate_execution, run_twap_execution, run_almgren_chriss_execution

eval_features, eval_raw_lob = eval_loader.sample_window(max(EXECUTION_STEPS, TWAP_SLICES + 1))

immediate = run_immediate_execution(eval_raw_lob, side=SIDE, parent_quantity=PARENT_QUANTITY)
twap = run_twap_execution(eval_raw_lob, side=SIDE, parent_quantity=PARENT_QUANTITY, n_slices=TWAP_SLICES)
almgren = run_almgren_chriss_execution(
    eval_raw_lob, side=SIDE, parent_quantity=PARENT_QUANTITY, n_slices=TWAP_SLICES,
    risk_aversion=AC_RISK_AVERSION, volatility=AC_VOLATILITY, temporary_impact=AC_TEMPORARY_IMPACT,
)

print(f"{'Baseline':<20} {'IS (bps)':>10} {'Filled':>10} {'Remaining':>12} {'Reward':>10}")
print("-" * 65)
for b in [immediate, twap, almgren]:
    print(
        f"{b.name:<20} {b.implementation_shortfall_bps:>10.2f} "
        f"{b.filled_qty:>10.0f} {b.remaining_inventory:>12.0f} {b.total_reward:>10.2f}"
    )

In [ ]:
from midmamba.eval import run_policy_evaluation
from contextlib import nullcontext

# Load the final checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
agent.load_state_dict(ckpt["model"])
agent.eval()

raw_eval_env = MidMambaExecutionEnv(
    eval_loader,
    execution_steps=EXECUTION_STEPS,
    initial_inventory=PARENT_QUANTITY,
    side=SIDE,
    fill_model="proportional",
    terminal_penalty_bps=TERMINAL_PENALTY_BPS,
    **_reward_kwargs,
)
eval_env = VecNormalize(raw_eval_env, norm_obs=True, norm_reward=False, gamma=GAMMA)
if "vec_normalize" in ckpt:
    eval_env.set_state(ckpt["vec_normalize"])
    print("VecNormalize stats restored from checkpoint")
elif hasattr(vec_env, "obs_rms"):
    eval_env.obs_rms = vec_env.obs_rms
    eval_env.ret_rms = vec_env.ret_rms
    print("VecNormalize stats copied from training env")
eval_env.training = False
amp_eval = torch.autocast(device_type="cuda", dtype=torch.bfloat16) if USE_AMP else nullcontext()

eval_result = run_policy_evaluation(
    agent, eval_env, n_episodes=EVAL_EPISODES,
    seq_len=SEQ_LEN, device=DEVICE, use_amp=USE_AMP,
)
eval_result.print_summary()
policy_shortfalls = eval_result.shortfalls_bps

print(f"\n{'Strategy':<20} {'IS (bps)':>10}")
print("-" * 32)
print(f"{'Immediate':<20} {immediate.implementation_shortfall_bps:>10.2f}")
print(f"{'TWAP':<20} {twap.implementation_shortfall_bps:>10.2f}")
print(f"{'Almgren-Chriss':<20} {almgren.implementation_shortfall_bps:>10.2f}")
print(f"{'Policy (mean)':<20} {np.mean(policy_shortfalls):>10.2f}")

In [ ]:
from midmamba.env import MBP10ExecutionEnv

# Record one deterministic policy episode
obs, info = eval_env.reset(seed=42)
obs_window = deque(
    [np.zeros_like(obs, dtype=np.float32)] * (SEQ_LEN - 1) + [obs.astype(np.float32)],
    maxlen=SEQ_LEN,
)
policy_traj = [info]
while True:
    obs_seq = torch.as_tensor(
        np.stack(obs_window)[None, :, :], dtype=torch.float32, device=DEVICE
    )
    with torch.no_grad(), amp_eval:
        action, _, _ = sample_squashed_normal(agent, obs_seq, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(
        action.squeeze(0).cpu().numpy()
    )
    obs_window.append(obs.astype(np.float32))
    policy_traj.append(info)
    if terminated or truncated:
        break

# Record one TWAP episode on the same raw_lob
twap_env = MBP10ExecutionEnv(
    eval_raw_lob, side=SIDE, parent_quantity=PARENT_QUANTITY,
    child_fraction=1.0 / float(TWAP_SLICES),
    end_index=len(eval_raw_lob) - 1,
)
_, twap_info = twap_env.reset()
twap_traj = [twap_info]
while True:
    _, _, terminated, truncated, twap_info = twap_env.step(1)
    twap_traj.append(twap_info)
    if terminated or truncated:
        break

# Plot
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

for name, traj in [("Policy", policy_traj), ("TWAP", twap_traj)]:
    steps = [t.get("step", t.get("row", i)) for i, t in enumerate(traj)]
    inv = [t.get("inventory", t.get("remaining_inventory")) for t in traj]
    is_bps = [t["implementation_shortfall_bps"] for t in traj]
    axes[0].plot(steps, inv, label=name)
    axes[1].plot(steps, is_bps, label=name)

mids = [t["mid_now"] for t in policy_traj]
p_steps = [t.get("step", i) for i, t in enumerate(policy_traj)]
axes[2].plot(p_steps, mids, color="black", linestyle="--", label="Mid Price")

axes[0].set_ylabel("Inventory")
axes[0].set_title("Execution Inventory Trajectory")
axes[0].legend()
axes[0].grid(True)

axes[1].set_ylabel("IS (bps)")
axes[1].set_title("Cumulative Implementation Shortfall")
axes[1].legend()
axes[1].grid(True)

axes[2].set_ylabel("Price")
axes[2].set_title("Market Mid Price")
axes[2].set_xlabel("Step")
axes[2].grid(True)

plt.tight_layout()
traj_path = RESULTS_DIR / "execution_trajectory.png"
plt.savefig(traj_path, dpi=150)
plt.show()
print(f"saved: {traj_path}")

In [ ]:
report = {
    "config": run_config,
    "baselines": {
        "immediate": immediate.to_dict(),
        "twap": twap.to_dict(),
        "almgren_chriss": almgren.to_dict(),
    },
    "policy": eval_result.summary(),
}

report_path = RESULTS_DIR / "colab_full_eval.json"
report_path.write_text(json.dumps(report, indent=2, default=str))

print("Drive outputs:")
print(f"  checkpoint:  {CHECKPOINT_PATH}")
print(f"  training:    {RESULTS_DIR / 'ppo_training_metrics.json'}")
print(f"  eval report: {report_path}")
print(f"  train plot:  {RESULTS_DIR / 'training_curves.png'}")
print(f"  traj plot:   {RESULTS_DIR / 'execution_trajectory.png'}")